In [1]:
#!pip install dataretrieval --break-system-packages
import pandas as pd
import requests

In [5]:
# Define station and parameters
API_KEY = "oW8binKFe14TK7EgAYq0PU2fhGkuZQYN4sAPCAgf"
STATION_ID = "USGS-03274615"
PARAMETER = "00065"  # 00060 = Discharge in cfs (cubic feet per second)
TIME_RANGE = "2023-01-01/2023-01-31"  # YYYY-MM-DD / YYYY-MM-DD

In [6]:
# 2. Endpoint URL
url = "https://api.waterdata.usgs.gov/ogcapi/v0/collections/daily/items"

# 3. Parameters including API Key
params = {
    "monitoring_location_id": STATION_ID,
    "parameter_code": PARAMETER,
    "time": TIME_RANGE,
    "api_key": API_KEY,  # Attach API key to unlock higher rate limits
}

In [8]:
# 4. Fetch Data
response = requests.get(url, params=params)
data = response.json()


In [13]:
data.get("features", [])

[{'type': 'Feature',
  'properties': {'time_series_id': 'de7bdccce770442d9fdcbce311deb033',
   'monitoring_location_id': 'USGS-03274615',
   'parameter_code': '00065',
   'statistic_id': '00003',
   'time': '2023-01-31',
   'value': '3.89',
   'unit_of_measure': 'ft',
   'approval_status': 'Approved',
   'qualifier': None,
   'last_modified': '2025-02-21T18:56:16.642560+00:00'},
  'id': '0704f048-2921-4b56-862c-a2d0a6abae98',
  'geometry': {'type': 'Point',
   'coordinates': [-84.7027777777778, 39.2161111111111]}},
 {'type': 'Feature',
  'properties': {'time_series_id': 'de7bdccce770442d9fdcbce311deb033',
   'monitoring_location_id': 'USGS-03274615',
   'parameter_code': '00065',
   'statistic_id': '00003',
   'time': '2023-01-29',
   'value': '3.46',
   'unit_of_measure': 'ft',
   'approval_status': 'Approved',
   'qualifier': None,
   'last_modified': '2025-02-21T18:56:16.642560+00:00'},
  'id': '07e6112d-e0bb-4903-9a10-ca6e09525374',
  'geometry': {'type': 'Point',
   'coordinates':

In [21]:
records = []
# Extract features from GeoJSON/OGC API response
for feature in data.get("features", []):
    props = feature.get("properties", {})
    feetVal = props.get("value")

    # Convert to m³/s
    feetVal = float(feetVal)
    if feetVal > 0:
        metersVal = feetVal * 0.3048 
        

        records.append(
            {
                "date": props.get("time"),
                "depth": metersVal,
                "qualifier": props.get("qualifier"),
            } 
        )

df = pd.DataFrame(records)
print(df.head())

         date     depth qualifier
0  2023-01-31  1.185672      None
1  2023-01-29  1.054608      None
2  2023-01-22  1.027176      None
3  2023-01-14  1.060704      None
4  2023-01-18  0.633984      None


In [24]:
df = df.set_index('date')
df = df.sort_index()
df

,depth,qualifier
date,,
2023-01-04,0.643128,None
2023-01-06,0.627888,None
2023-01-12,0.396240,None
2023-01-14,1.060704,None
2023-01-18,0.633984,None
2023-01-19,0.926592,None
2023-01-22,1.027176,None
2023-01-26,1.368552,None
2023-01-29,1.054608,None


In [ ]:

if response.status_code == 200:
    data = response.json()
    records = []

    # Extract features from GeoJSON/OGC API response
    for feature in data.get("features", []):
        props = feature.get("properties", {})
        cfs_val = props.get("value")

        # Convert to m³/s
        m3s_val = cfs_val * 0.0283168 if cfs_val is not None else None

        records.append(
            {
                "date": props.get("time"),
                "discharge_cfs": cfs_val,
                "discharge_m3s": m3s_val,
                "qualifier": props.get("qualifier"),
            }
        )

    df = pd.DataFrame(records)
    print(df.head())
else:
    print(f"Error {response.status_code}: {response.text}")